ret: normal ret, setting pc to return address (ra).

we can' use it to go from kernel mode to user mode, because:

```
ret    # pc = ra, privilege unchanged
```

Privilege needs to update otherwise user process can have kernel privilege.

How?


| Register / Field | Purpose |
|---|---|
| `sepc` | Tells the CPU where to resume execution |
| `sstatus.SPP` | Tells the CPU which privilege level `sret` should return to |
| `sret` | Performs the privilege-level transition and jumps to the address stored in `sepc` |

Suppose we want to transition from Kernel Mode to User Mode and start executing at address `0x1000`:

```
# Set the target privilege to User Mode.
csrr t0, sstatus
li t1, ~(1 << 8)
and t0, t0, t1

csrw sstatus, t0

# Set the target instruction address.
li t0, 0x1000
csrw sepc, t0

# Enter User Mode.
sret
```

---

the whole flow:
```
scheduler()
    |
    | swtch()
    v
forkret()               Supervisor Mode
    |
    | prepare_return()
    |   Set SPP = 0
    |   Set sepc = user entry point
    |   Set stvec = uservec
    |
    v
userret                 Supervisor Mode
    |
    | Switch to User Page Table
    | Restore User Registers
    | sret
    v
User Program            User Mode
```
